In [1]:
import logging
from pathlib import Path
from typing import Any, Dict, Tuple
import os
import numpy as np
import PIL
import SimpleITK as sitk
from PIL.Image import Resampling
from skimage.measure import find_contours

from skimage.transform import resize

logger = logging.getLogger(__name__)

In [2]:
def sitk_load(filepath: str | Path) -> Tuple[np.ndarray, Dict[str, Any]]:
    """Loads an image using SimpleITK and returns the image and its metadata.

    Args:
        filepath: Path to the image.

    Returns:
        - ([N], H, W), Image array.
        - Collection of metadata.
    """
    # Load image and save info
    image = sitk.ReadImage(str(filepath))
    info = {"origin": image.GetOrigin(), "spacing": image.GetSpacing(), "direction": image.GetDirection()}

    # Extract numpy array from the SimpleITK image object
    im_array = np.squeeze(sitk.GetArrayFromImage(image))

    return im_array, info

In [3]:
def read_txt(filename):
        with open(filename) as file:
            lines = [line.rstrip() for line in file]
        return lines

In [4]:
# CAMUS ground-truth labels are integer class maps: 0=background,
# 1=LV endocardium, 2=myocardium, 3=left atrium.
NUM_CLASSES = 4


def save_preproced_image(img, size, path):
    img = resize(img, size, anti_aliasing=True)
    img = (img - np.min(img)) / (np.max(img) - np.min(img))  # Normalize to [0, 1]
    np.save(path, img.astype(np.float32))


def save_preproced_label(label, size, path, num_classes=NUM_CLASSES):
    """Resize a class-index label map and store it one-hot encoded.

    Uses nearest-neighbor resizing (order=0, no anti-aliasing) so class
    boundaries stay crisp integers instead of being blurred into
    fractional values by an interpolating resize — anti-aliasing is only
    correct for continuous-valued images, not label maps. The resized
    index map is then one-hot encoded before saving.

    Saves an array of shape (num_classes, H, W), channel 0 = background.
    """
    label = resize(label, size, order=0, anti_aliasing=False, preserve_range=True)
    label = np.rint(label).astype(np.int64)
    onehot = np.eye(num_classes, dtype=np.float32)[label]   # (H, W, C)
    onehot = np.moveaxis(onehot, -1, 0)                      # (C, H, W)
    np.save(path, onehot)

In [5]:
source_root  ='../../data/Camus/camus/Resources'
target_root ='../../data/Camus/preprocessed_data'
nifti_folder =os.path.join(source_root,'database_nifti')
database_split = os.path.join(source_root,'database_split')
target_size = (256,256)


split_paths = {
    'train': os.path.join(database_split, 'subgroup_training.txt'),
    'val': os.path.join(database_split, 'subgroup_validation.txt'),
    'test': os.path.join(database_split, 'subgroup_testing.txt')
}
patients = {split: read_txt(path) for split, path in split_paths.items()}

In [6]:
for mode in patients:
    mode_folder = os.path.join(target_root, mode)
    os.makedirs(mode_folder, exist_ok=True)
    for p in patients[mode]:
        patient_source_folder = os.path.join(nifti_folder,  p)
        patient_target_folder = os.path.join(mode_folder,  p)
        os.makedirs(patient_target_folder, exist_ok=True)

        for file in os.listdir(patient_source_folder):
            if file.endswith('_gt.nii.gz'):
                split = file.split('_')
                view = split[1]
                frame = split[2]
                view_target_path = os.path.join(patient_target_folder, view)
                os.makedirs(view_target_path, exist_ok=True)
                
                split_target_path  = os.path.join(view_target_path, frame)
                os.makedirs(split_target_path, exist_ok=True)

                annotation_file  =file
                patient_file  =file.replace('_gt', '')
                annotation = sitk_load(os.path.join(patient_source_folder, annotation_file))[0]
                patient = sitk_load(os.path.join(patient_source_folder, patient_file))[0]
                if frame =='half':
                    assert annotation.shape == patient.shape
                    for i in range(annotation.shape[0]):
                        slice_folder = os.path.join(split_target_path, str(i))
                        os.makedirs(slice_folder, exist_ok=True)
                        slice_path = os.path.join(slice_folder, str(i) + '.nii.gz')
                        slice_annotation_path = os.path.join(slice_folder, str(i) + '_gt.nii.gz')

                        save_preproced_image(patient[i], target_size, slice_path)
                        save_preproced_label(annotation[i], target_size, slice_annotation_path)

                else:
                    slice_folder = os.path.join(split_target_path, '0')
                    os.makedirs(slice_folder, exist_ok=True)
                    patient_save_path = os.path.join(slice_folder, 'patient.nii.gz')
                    annotation_save_path = os.path.join(slice_folder, '_gt.nii.gz')

                    save_preproced_image(patient, target_size, patient_save_path)
                    save_preproced_label(annotation, target_size, annotation_save_path)